In [74]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [75]:
train_df = pd.read_parquet('train_final.parquet')
test_df = pd.read_parquet('test_final.parquet')
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
train_df.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,5216127,3,1,0,0,0,0,0.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,FTP-Patator
1,21,20,1,1,0,0,0,0,0.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,FTP-Patator
2,21,38,1,1,0,0,0,0,0.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,FTP-Patator
3,21,80,1,1,0,0,0,0,0.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,FTP-Patator
4,21,68,1,1,0,0,0,0,0.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,FTP-Patator


In [76]:
from utils import handle_values
train_df = handle_values(train_df.copy())
test_df = handle_values(test_df.copy())

In [77]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
train_df[' Label']= encoder.fit_transform(train_df[' Label'])
train_df[' Label'].value_counts()

test_df[' Label']= encoder.transform(test_df[' Label'])
test_df[' Label'].value_counts()

 Label
9    127144
0    120000
2    102422
1       983
Name: count, dtype: int64

In [78]:
X_train = train_df.drop(' Label',axis=1)
y_train = train_df[' Label']
X_test = test_df.drop(' Label',axis=1)
y_test = test_df[' Label']

In [79]:
corr = X_train.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]

In [80]:
X_train =X_train.drop(to_drop,axis=1)
X_test =X_test.drop(to_drop,axis=1)

In [81]:
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
X_train_np = X_train.values
X_test_np = X_test.values


imputer = SimpleImputer(strategy='constant',fill_value=0)
scaler = RobustScaler()


X_train_np = imputer.fit_transform(X_train_np)
X_train_np = scaler.fit_transform(X_train_np)


X_test_np = imputer.transform(X_test_np)
X_test_np = scaler.transform(X_test_np)


train_final = pd.DataFrame(
    X_train_np, 
    columns=X_train.columns
)
test_final = pd.DataFrame(
    X_test_np, 
    columns=X_test.columns
)

In [82]:
train_final[' Label'] = train_df[' Label']
test_final[' Label'] = test_df[' Label']

In [83]:
train_final.shape

(156024, 45)

In [84]:
y_train.value_counts()

 Label
9     31786
0     30000
4     30000
2     25605
3     10293
7      7938
10     5897
6      5796
5      5499
11     1507
1       983
8       720
Name: count, dtype: int64

In [85]:
import lightgbm as lgb
lgbm_gpu = lgb.LGBMClassifier(
    objective='multiclass', 
    num_class=len(np.unique(y_train)),
    n_estimators=100,            
    device='cpu',           
    n_jobs=-1,                   
    random_state=42,
    min_child_samples=1,
    max_depth=4
)

print("Starting LightGBM training for feature importance using the GPU...")
lgbm_gpu.fit(X_train_np, y_train)
print("Training complete and significantly faster!")

Starting LightGBM training for feature importance using the GPU...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012890 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6423
[LightGBM] [Info] Number of data points in the train set: 156024, number of used features: 36
[LightGBM] [Info] Start training from score -1.648812
[LightGBM] [Info] Start training from score -5.067156
[LightGBM] [Info] Start training from score -1.807222
[LightGBM] [Info] Start training from score -2.718546
[LightGBM] [Info] Start training from score -1.648812
[LightGBM] [Info] Start training from score -3.345444
[LightGBM] [Info] Start training from score -3.292842
[LightGBM] [Info] Start training from score -2.978348
[LightGBM] [Info] Start training from score -5.378514
[LightGBM] [Info] Start training from score -1.590984
[LightGBM] [Info] Start training from s

In [86]:

# --- C. Get Importance Scores ---
feature_importances = pd.Series(lgbm_gpu.feature_importances_, index=X_train.columns)
feature_importances

 Destination Port              1956
 Flow Duration                  984
 Total Fwd Packets              586
Total Length of Fwd Packets     801
 Fwd Packet Length Max          487
 Fwd Packet Length Min           88
 Fwd Packet Length Mean         331
Bwd Packet Length Max           576
 Bwd Packet Length Min          138
Flow Bytes/s                    556
 Flow Packets/s                 312
 Flow IAT Mean                  345
 Flow IAT Std                   388
 Flow IAT Min                  1548
 Fwd IAT Mean                   443
 Fwd IAT Min                   1144
Bwd IAT Total                   211
 Bwd IAT Mean                   146
 Bwd IAT Std                    184
 Bwd IAT Min                    421
Fwd PSH Flags                    79
 Bwd PSH Flags                    0
 Fwd URG Flags                   11
 Bwd URG Flags                    0
 Bwd Packets/s                  725
 Min Packet Length               89
FIN Flag Count                   89
 RST Flag Count             

In [87]:
feature_importances_series = pd.Series(
    lgbm_gpu.feature_importances_, 
    index=X_train.columns
).sort_values(ascending=False)

# 2. Print the top 20 to decide on a cutoff

imp_features = feature_importances_series.head(20).index.tolist()
imp_features

[' Destination Port',
 ' Flow IAT Min',
 ' Init_Win_bytes_backward',
 'Init_Win_bytes_forward',
 ' Fwd IAT Min',
 ' Flow Duration',
 'Total Length of Fwd Packets',
 ' Bwd Packets/s',
 ' Total Fwd Packets',
 'Bwd Packet Length Max',
 'Flow Bytes/s',
 ' Fwd Packet Length Max',
 ' Fwd IAT Mean',
 ' min_seg_size_forward',
 ' Bwd IAT Min',
 ' Flow IAT Std',
 ' Flow IAT Mean',
 ' Fwd Packet Length Mean',
 ' Flow Packets/s',
 ' URG Flag Count']

In [88]:
# 1. Filter the importance scores to include ONLY the top 20 features
feature_importances_20 = feature_importances_series[imp_features]

# 2. Calculate the total importance score for these 20 features
# This sum will be used as the denominator for normalization
total_importance = feature_importances_20.sum()

# 3. Normalize the scores to get the percentage weight
# (Divide each score by the total sum and multiply by 100)
normalized_importance = (feature_importances_20 / total_importance) * 100

# 4. Sort and format the results for clear visualization
normalized_importance_sorted = normalized_importance.sort_values(ascending=False).round(2)

print("\n--- Normalized Feature Importance (Weight %) ---")
print(normalized_importance_sorted)


--- Normalized Feature Importance (Weight %) ---
 Destination Port              12.93
 Flow IAT Min                  10.24
 Init_Win_bytes_backward        9.59
Init_Win_bytes_forward          8.78
 Fwd IAT Min                    7.56
 Flow Duration                  6.51
Total Length of Fwd Packets     5.30
 Bwd Packets/s                  4.79
 Total Fwd Packets              3.87
Bwd Packet Length Max           3.81
Flow Bytes/s                    3.68
 Fwd Packet Length Max          3.22
 Fwd IAT Mean                   2.93
 min_seg_size_forward           2.86
 Bwd IAT Min                    2.78
 Flow IAT Std                   2.57
 Flow IAT Mean                  2.28
 Fwd Packet Length Mean         2.19
 Flow Packets/s                 2.06
 URG Flag Count                 2.05
dtype: float64


In [89]:
# from sklearn.feature_selection import SelectKBest, f_classif
# from sklearn.impute import SimpleImputer
# imputer = SimpleImputer(strategy='mean')
# X_imputed = imputer.fit_transform(X_train)

# # Determine the number of columns (features) in your DataFrame
# num_columns = train_df.shape[1]

# # Selecting an appropriate K
# k = min(20, num_columns)  # Will adjust as needed

# # Initialize SelectKBest with the scoring function
# k_best = SelectKBest(score_func=f_classif, k=k)

# # Fit and transform the imputed data to select the top 10 features
# X_new = k_best.fit_transform(X_imputed, y_train)

# # Get the boolean mask of selected features
# selected_features_mask = k_best.get_support()
# elected_feature_names = X_train.columns[selected_features_mask]

# elected_feature_names

In [90]:
final_features = imp_features + [' Label']

train_df_preprocessed = train_final[final_features]
test_df_preprocessed = test_final[final_features]

In [91]:
config_data = {
    'project_name': "Network Anomaly Detection",
    'feature_engineering': {
        'selected_features': imp_features
    }
}

In [92]:
import yaml

with open('config.yaml',"a") as f:
    yaml.dump(config_data,f,default_flow_style=False)

In [97]:
train_df_preprocessed.to_parquet("final_df/train_df_preprocessed.parquet")
test_df_preprocessed.to_parquet("final_df/test_df_preprocessed.parquet")

In [94]:
test_df_preprocessed.head()

,Destination Port,Flow IAT Min,Init_Win_bytes_backward,Init_Win_bytes_forward,Fwd IAT Min,Flow Duration,Total Length of Fwd Packets,Bwd Packets/s,Total Fwd Packets,Bwd Packet Length Max,...,Fwd Packet Length Max,Fwd IAT Mean,min_seg_size_forward,Bwd IAT Min,Flow IAT Std,Flow IAT Mean,Fwd Packet Length Mean,Flow Packets/s,URG Flag Count,Label
0,22.038567,0.500000,0.466102,0.973298,1.155,4.747787,0.948718,-0.000144,1.000000,0.057551,...,1.333333,2.238454,0.0,14.155556,1.411733,1.507873,0.625604,-0.000845,0.0,1
1,22.038567,997.810345,0.466102,-0.027186,-0.015,-0.012115,-0.083333,0.001432,-0.333333,-0.001381,...,-0.060606,-0.001402,0.0,0.000000,-0.009776,0.002914,-0.152174,0.000234,1.0,1
2,22.038567,0.982759,1.004237,0.247608,3.710,-0.006583,0.576923,0.002002,0.166667,0.028085,...,0.779221,0.013513,-1.0,37.800000,0.004855,-0.012896,0.967391,0.000871,0.0,1
3,22.038567,0.637931,1.093220,0.247608,0.365,-0.011549,5.910256,0.004086,0.166667,0.028085,...,7.982684,0.006189,-1.0,22.600000,-0.002474,-0.017414,10.010870,0.002536,0.0,1
4,4.865014,0.413793,1.084746,-0.027186,-0.015,-0.016693,-0.064103,2.190329,-0.333333,0.000000,...,-0.034632,-0.001402,-1.0,0.000000,-0.009776,-0.022078,-0.021739,1.499171,1.0,1


In [96]:
import joblib

joblib.dump(scaler,'final_df/fitted_robust_scaler.joblib')
joblib.dump(imputer,'final_df/fitted_robust_imputer.joblib')

['final_df/fitted_robust_imputer.joblib']